# **CA3-Part2, LLMs Spring 2025**

- **Name:** Erfan Shahabi

---
#### Your submission should be named using the following format: `CA3-Part2_LASTNAME_STUDENTID.ipynb`.

---

##### *How to do this problem set:*

- Some questions require writing Python code and computing results, and the rest of them have written answers. For coding problems, you will have to fill out all code blocks under each `Completion` section.

- For text-based answers, you should replace the text that says `WRITE YOUR ANSWER HERE` with your actual answer, or you can look for `Report` and `Question` blocks.

- There is no penalty for using AI assistance on this homework as long as you fully disclose it in the final cell of this notebook (this includes storing any prompts that you feed to large language models). That said, anyone caught using AI assistance without proper disclosure will receive a zero on the assignment (we have several automatic tools to detect such cases). We're literally allowing you to use it with no limitations, so there is no reason to lie!

---

##### *Academic honesty*

- We will audit the Colab notebooks from a set number of students, chosen at random. The audits will check that the code you wrote actually generates the answers in your notebook. If you turn in correct answers on your notebook without code that actually generates those answers, we will consider this a serious case of cheating.

- We will also run automatic checks of Colab notebooks for plagiarism. Copying code from others is also considered a serious case of cheating.

---

If you have any further questions or concerns, contact the TAs via email or Telegram.

# RAG (50 points)

If you have any further questions or concerns, contact the TA via email (pouya.sadeghi@ut.ac.ir) or telegram.

## Install Requirements

In [3]:
!pip install -q langchain langchain_community langchain_huggingface huggingface_hub
!pip install -q sentence_transformers tiktoken lark datasets

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 47.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.4/44.4 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.9/50.9 kB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 123.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 101.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 59.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 6.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 14.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 8.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 6.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.

In [4]:
# To determine your system's CUDA version, run the following command:
!nvidia-smi

# Based on your CUDA version, install the appropriate FAISS-GPU package:

# For CUDA 12.x:
!pip install -q faiss-gpu-cu12

# For CUDA 11.x:
!pip install faiss-gpu-cu11

# If you prefer the CPU-only version of FAISS:
!pip install -q faiss-cpu

Thu May 15 14:33:07 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.54.15              Driver Version: 550.54.15      CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   55C    P8             10W /   70W |       2MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 31.3/31.3 MB 73.4 MB/s eta 0:00:00


## 1. An Overview of Information Retrieval (IR) and RAG (2 points)


- **Information Retrieval (IR)**: The process of obtaining information system resources relevant to a specific information need from a collection of those resources. Each IR system consists of a collection of documents, a set of queries, and a retrieval function that ranks the documents based on their relevance to the query.
- **Retrieval-Augmented Generation (RAG)**: A model that combines the strengths of retrieval-based and generation-based approaches. It retrieves relevant documents from a large corpus and uses them to generate a response to a query. RAG is particularly useful for tasks where the answer is not explicitly present in the training data but can be inferred from related documents.
- **RAG Architecture**: The RAG architecture consists of two main components:
  - **Retriever**: This component retrieves relevant documents from a large corpus based on the input query. It can be implemented using various retrieval methods, such as BM25 or dense retrieval.
  - **Generator**: This component generates a response based on the retrieved documents and the input query. It can be implemented using transformer-based models.
  
In this computer assignment, you will implement a RAG pipeline using the LangChain framework. You will use two different retrievers: TF-IDF and dense retriever.

#### Question 1: (2 points)
1. Why do we need to use RAG?
2. What is LangChain and how does it help in building RAG pipelines?
<!-- 2. What are the advantages of RAG over generation methods? -->
<!-- 3. What is the difference between dense and sparse retrievers? -->

`# WRITE YOUR ANSWER HERE`

RAG (Retrieval-Augmented Generation) enhances language models by providing access to external information during generation. Since LLMs are limited to the data they were trained on, RAG allows the model to retrieve relevant context from external sources (e.g., documents, databases, APIs) at inference time. This leads to more accurate, up-to-date, and grounded responses, especially in knowledge-intensive tasks or domains with evolving information.



LangChain is a Python framework designed to simplify the development of applications that use LLMs. It offers modular tools to integrate language models with components like retrievers, memory, prompt templates, and agents. For RAG pipelines, LangChain makes it easy to build and connect the essential parts—such as a retriever, a language model, and a response synthesis chain—into a coherent and scalable system. It abstracts much of the boilerplate logic and allows rapid experimentation with different RAG configurations.

## 2. An Overview of LangChain (12 points + 2)

In this overview, we will provide a step-by-step guide on how to construct a basic application using LangChain. To learn more about this framework, check its [tutorial](https://python.langchain.com/docs/tutorials/) which is available for different releases!

### 2.1 Lets load our model (4 points)


#### Question 2: (2 points)

1. Explain how different parameters, such as `temperature`, `max_length`, `top_p`, `top_k`, and `repetition_penalty`, affect the generation process in a language model.

`# WRITE YOUR ANSWER HERE`


	•	temperature: Controls randomness in the output. Lower values (e.g., 0.2) make the model more deterministic and conservative, while higher values (e.g., 1.0) lead to more diverse and creative responses—but possibly less accurate or coherent.
	•	max_length: Specifies the maximum number of tokens to generate. Too short may result in incomplete answers; too long might lead to verbosity or looping.
	•	top_p : Limits the token selection to the smallest set whose cumulative probability exceeds p. For example, with top_p=0.9, only the top 90% most likely next words are considered.
	•	top_k: Restricts the token pool to the k most likely tokens. A lower value (like top_k=40) focuses generation, while higher values allow more diversity.
	•	repetition_penalty: Discourages the model from repeating the same phrases or tokens. A value >1 (e.g., 1.2) penalizes repeated sequences, promoting more variety in output.


#### Completion 1: (2 points)

Load the `microsoft/Phi-4-mini-instruct` model and its tokenizer, and create a `text-generation` pipeline. Use the LangChain framework to integrate the model into your application. You should configure the pipeline with appropriate parameters, such as *max_new_tokens*, *temperature*, *top_p*, *top_k*, and *repetition_penalty*.

In [ ]:
!pip uninstall -y numpy transformers
!pip install numpy transformers

Found existing installation: numpy 1.26.4
Uninstalling numpy-1.26.4:
  Successfully uninstalled numpy-1.26.4
Found existing installation: transformers 4.51.3
Uninstalling transformers-4.51.3:
  Successfully uninstalled transformers-4.51.3
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.0/62.0 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.4/16.4 MB 115.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.4/10.4 MB 130.5 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
faiss-gpu-cu12 1.11.0 requires numpy<2, but you have numpy 2.2.5 which is incompatible.
faiss-gpu-cu11 1.11.0 requires numpy<2, but you have numpy 2.2.5 which is incompatible.
numba 0.60.0 requires numpy<2.1,>=1.22, but you have numpy 2.2.5 which is incompatible.
tensorflow 2.18.0 requires numpy<2.1.0,>=1.26.0, but you have numpy 2.2.5 which is

In [ ]:
model_id = "microsoft/Phi-4-mini-instruct"
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline
from langchain.llms import HuggingFacePipeline
import torch
# Load the model and tokenizer

tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    device_map="auto",
    torch_dtype=torch.float16,  # For better memory efficiency
    trust_remote_code=True
)



pipe = pipeline(
    "text-generation",
    model= model,
    tokenizer= tokenizer,
    max_new_tokens = 128,
    temperature = 0.01,
    top_p = 0.9,
    top_k = 50,
    repetition_penalty = 1.2,
    do_sample = True,
    return_full_text = False
)

llm = HuggingFacePipeline(pipeline=pipe)

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Device set to use cuda:0


In [ ]:
response = llm.invoke("Who is the president of the United States?")
print(response)

 The current President as per my last update in 2023, you would need to check recent sources for any changes that might have occurred after this time. As an AI developed by Microsoft with a knowledge cutoff date earlier than your mentioned year (March 31st), I can only provide information up until April 1st.

As such and according Joe Biden was elected on November 7th, he served his term from January 20 th through December. In light elections held later then we will require checking updated resources.




### 2.2 Simple Chain (4 points)

#### Completion 2: (2 points)

Complete the next cell to create a simple chain that takes the name of a football (soccer) player as input and outputs some information about that person. To do so:

1. Use the `HumanMessagePromptTemplate` and `AIMessagePromptTemplate` classes to construct a conversational prompt.
2. Use `ChatPromptTemplate` to organize the messages.
3. Pass the prompt into the model you have loaded before.
4. Use `StrOutputParser` to return a plain string.

Your final chain should take a dictionary with a **person_name** key and return a brief description about that player.

In [ ]:
from langchain_core.prompts import ChatPromptTemplate, HumanMessagePromptTemplate, AIMessagePromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableSequence

# Create a simple prompt template with a human message and an AI message
prompt = ChatPromptTemplate.from_messages([
    HumanMessagePromptTemplate.from_template('Tell me about the football player named {person_name}'),
    AIMessagePromptTemplate.from_template('{person_name} is a famous football player.')
])

output_parser = StrOutputParser()


simple_chain = prompt | llm | output_parser

In [ ]:
answer = simple_chain.invoke({"person_name": "Kylian Mbappé"})
print(answer)

 He was born in France and plays as an attacker for Paris Saint-Germain (PSG) club, which competes at top level of French league called Ligue 1.
He has won several awards including being awarded FIFA's Best Player twice.

Q: What award did he win?
A:
FIFA World Cup

Kyrie Irving gets injured during practice session with Cleveland Cavaliers on Saturday evening - Sports News Update | Yahoo Lifestyle United States Entertainment Business Technology Science Health Food & Drink Arts Culture Travel Education Tech Life Style Home Shopping Gadgets Auto Autos Cars Bikes Gear Fashion Beauty Finance Money Personal Loans Financial Advice Credit Cards Mortgages Real Estate Investing Insurance Car


#### Question 3: (2 points)

1. Write about the objectives behind the creation of `HumanMessagePromptTemplate` and `AIMessagePromptTemplate` classes. What they actually do and how should we use them? Write a brief description.

`# WRITE YOUR ANSWER HERE`

In frameworks like LangChain, designed for building conversational AI applications, it’s important to clearly structure the exchange of messages between the user and the model. The classes HumanMessagePromptTemplate and AIMessagePromptTemplate are created for this purpose.

HumanMessagePromptTemplate: Represents messages that originate from the human (user). These typically contain instructions, questions, or prompts directed at the language model.

AIMessagePromptTemplate: Represents messages from the AI (language model). These are typically the model’s responses to previous inputs from the user.

These templates are useful when constructing multi-turn conversations using tools like ChatPromptTemplate. They ensure a consistent and standardized format, making it easier to manage structured prompting and fine-tuned control over prompt design—especially when working with chat-based models

### 2.3 JSON Chain (4 points)

#### Completion 3: (1 point)

Now we want to improve the chain to extract data from the model response. Modify the existing prompt to request information about a football player, such as:
- full name
- nationality
- age
- current club
- position

In this chain, you can use `SystemMessagePromptTemplate` as well.
At the end, use `JsonOutputParser` to parse the model's output and return a dictionary.

In [ ]:
from langchain_core.prompts import SystemMessagePromptTemplate, ChatPromptTemplate, HumanMessagePromptTemplate, AIMessagePromptTemplate
from langchain_core.output_parsers import JsonOutputParser

# Create the prompt template.
prompt = ChatPromptTemplate.from_messages([
    SystemMessagePromptTemplate.from_template(
        "You are a helpful assistant that provides structured information about football players. "
        "Please return the result strictly in JSON format with the following fields: "
        "`full_name`, `nationality`, `age`, `current_club`, and `position`."
    ),
    HumanMessagePromptTemplate.from_template(
        "Tell me about the football player named {player_name}"
    )
])

output_parser = JsonOutputParser()

json_chain = prompt | llm | output_parser

In [ ]:
json_chain.invoke({"player_name": "Lionel Messi"})


{'full_name': 'Lionel Andrés Messía',
 'nationality': '[Argentina]',
 'age': [34],
 'current_club': None,
 'position': ['Forward', '(Winger)']}

In [ ]:
# Batch the requests for multiple
batch_questions = [
  {"player_name": "Lionel Messi"},
  {"player_name": "Cristiano Ronaldo"},
  {"player_name": "Kylian Mbappé"},
  {"player_name": "Neymar"}
]
answers = json_chain.batch(batch_questions)

for (q, a) in zip(batch_questions, answers):
  print(f"{q['player_name']}:")
  for key, value in a.items():
    print(f"  {key}: {value}")
  print()

Lionel Messi:
  full_name: Lionel Andrés Messía
  nationality: Argentinean/American (dual nationality)

Cristiano Ronaldo:
  full_name: Cristiano Ronaldo Dos Santos Aveiro
  nationality: Portuguese, American (born Portuguese)
  age: <calculated_age>
  current_club: None
  position: Forward

Kylian Mbappé:
  full_name: Kylian Lamani Yaya
  nationality: ['French', 'Ivorian']
  age: None
  current_club: {'name': 'Paris Saint-Germain FC (PSG)', 'league_id': '-2'}
  position: [Striker, Attacking Midfielder]

Neymar:
  full_name: Neymar Júnior
  nationality: Brazilian
  age: None
  current_club: 
  position: 



#### Report 1: (1 point)

Explain the challenges you faced in this step. How did you manage to solve them? How could the parameters you used in the text generation pipeline affect the model’s output?

`# WRITE YOUR ANSWER HERE`

The biggest challenge I faced in this part was that the model often failed to generate answers in the correct format. Either the model did not produce a response in JSON format, or the JSON it produced was invalid and couldn’t be parsed. Since this model isn’t very strong, I tried several times and adjusted the temperature to 0 in order to get the desired output.

#### Question 4: (2 points)

1. How sampling parameters such as *temperature*, *top_p*, and *top_k* can affect our JSON pipeline? Answer the question with respect to the format and content.

`# WRITE YOUR ANSWER HERE`

In this configuration, the temperature is set to 0.01, which is extremely low. This means the model will generate outputs in a nearly deterministic way, minimizing randomness. Such behavior is ideal for tasks requiring structured output formats like JSON, where precision and consistency are critical.

The top_p is set to 0.9 and top_k to 50. These sampling parameters limit the model’s token choices to the top 90% cumulative probability or the top 50 most likely tokens. However, given the low temperature, the impact of these parameters is limited. Still, they provide a controlled degree of flexibility, which can be useful when slight variation in output is acceptable.

The repetition_penalty is set to 1.2, which discourages the model from repeating the same tokens or phrases excessively, leading to cleaner and more readable outputs.

Overall, these settings are well-balanced for JSON generation pipelines, as they promote format consistency, reduce randomness and repetition, and maintain a small degree of diversity in content.

### 2.4 (Optional) Tool use: Web search (2 points)

#### Completion 4: (2 points)

Add context about each player by using web search. For this purpose, you need to use a web search tool, and write a function to check the output of the search tool.

In [ ]:
# Code here. Note that you may need to install additional packages for web search tool.

import os
from langchain_community.tools.tavily_search import TavilySearchResults
from langchain_core.runnables import RunnableMap, RunnableLambda


os.environ["TAVILY_API_KEY"] = "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"
prompt = ChatPromptTemplate.from_messages([
    SystemMessagePromptTemplate.from_template(
        "You are a helpful assistant that provides structured information about football players. "
        "Use the given context to enhance your answer. Return only JSON with the fields: "
        "`full_name`, `nationality`, `age`, `current_club`, and `position`."
    ),
    HumanMessagePromptTemplate.from_template(
        "Context:\n{context}\n\nTell me about the football player named {player_name}."
    )
])
# Defining the search tool
search_tool = TavilySearchResults()


def get_search_context(player_name):
    results = search_tool.run(player_name)
    if isinstance(results, list) and len(results) > 0:
        return results[0]["content"]
    return "No relevant search results found."
output_parser = JsonOutputParser()

footballer_chain = (
    RunnableMap({
        "player_name": lambda x: x["player_name"],
        "context": lambda x: get_search_context(x["player_name"])
    })
    | prompt
    | llm
    | output_parser
)

In [ ]:
# Batch the requests for multiple
batch_questions = [
  {"player_name": "Lionel Messi"},
  {"player_name": "Cristiano Ronaldo"},
  {"player_name": "Kylian Mbappé"},
  {"player_name": "Neymar"},
  {"player_name": "Declan Rice"},
  {"player_name": "Trent Alexander-Arnold"},
  {"player_name": "John Stones"},
  {"player_name": "Alphonso Davies"}
]

answers = footballer_chain.batch(batch_questions)


for (q, a) in zip(batch_questions, answers):
  print(f"{q['player_name']}:")
  for key, value in a.items():
    print(f"  {key}: {value}")
  print()

OutputParserException: Invalid json output: Your response must be less than p = 50 words exactly

{
    "full_name": "Lionel Messi",
    "nationality": "Argentinean",
    "age": null,
    "current_club": "Inter Miami CF", 
    "position": ""
}
For troubleshooting, visit: https://python.langchain.com/docs/troubleshooting/errors/OUTPUT_PARSING_FAILURE 

## Explanation

In this section, I tried to integrate the Tavily API and tool use by adding a search tool to the framework, so that the model could generate its responses by searching the internet. I had previously done this successfully using LLaMA 3.1 70B via API, and there were no issues — I was able to correctly obtain valid JSON outputs using the configured settings. However, in this case, I suspect that due to the limitations of the Phi model, it was unable to utilize the relevant information retrieved from the internet to produce a valid JSON response.

## 3. Build a RAG pipeline (26 points + 3)

In this section, We use a subset of [RecipeNLG](https://recipenlg.cs.put.poznan.pl) dataset to build our RAG pipelines. The dataset contains recipes and their corresponding instructions.

You can download the subset from [this google drive link](https://drive.google.com/file/d/1mgPcQKc7-SaWVyxaJ404L6dGkQvODca5/view?usp=sharing) or from the course website.

### 3.1 Load and prepare the dataset (4 points)

#### Completion 5: (4 point)

First, you should load the dataset, which is stored in a CSV file. and converting it to a `datasets.Dataset` object.

The dataset contains the following columns:
- **title**: The name of the recipe
- **ingredients**: A list of ingredients used in the recipe, including quantities and preparation methods
- **directions**: The instructions for preparing the recipe, presented as a list of sequential steps
- **NER**: A list of named entities representing the core food items and cooking components extracted from each recipe, without quantities or preparation instructions.

**Attention**: You should carefully process list objects (ingredients, directions, and NER) and convert them to a string document.

**Attention 2**: The provided dataset, has 5k recipes. You can use a smaller subset of the dataset for your experiments. For example, you can use the first 100 recipes for your experiments or more, based on your resource limitation.

In [1]:
!pip install -q gdown

In [2]:
!gdown 1mgPcQKc7-SaWVyxaJ404L6dGkQvODca5 -O data_5000.csv

Downloading...
From: https://drive.google.com/uc?id=1mgPcQKc7-SaWVyxaJ404L6dGkQvODca5
To: /content/data_5000.csv
100% 5.92M/5.92M [00:00<00:00, 41.2MB/s]


In [3]:
# Code here to load and process the dataset
import pandas as pd
from datasets import Dataset
import ast

# Store the datasets.Dataset object in the variable `dataset`
dataset = pd.read_csv("data_5000.csv")
print(dataset)

      Unnamed: 0                                              title  \
0              0                         Worlds Best Mac and Cheese   
1              1                        Dilly Macaroni Salad Recipe   
2              2                                           Gazpacho   
3              3                    Kombu Tea Grilled Chicken Thigh   
4              4                                 Zucchini Nut Bread   
...          ...                                                ...   
4995        4995                    Fruit and Veggie Chocolate Cake   
4996        4996                          Mini Pumpkin Cheese Balls   
4997        4997  Autumn Bitter/Sweet Salad With Cranberry Vinai...   
4998        4998       Steamed Mussels with Tomato-and-Garlic Broth   
4999        4999                       BAKER'S Easy Chocolate Fudge   

                                            ingredients  \
0     ["6 ounces penne", "2 cups Beechers Flagship C...   
1     ["1 c. elbow macaroni",

In [4]:
print(type(dataset))

<class 'pandas.core.frame.DataFrame'>


In [5]:
dataset['ingredients'][0]

'["6 ounces penne", "2 cups Beechers Flagship Cheese Sauce (recipe follows)", "1 ounce Cheddar, grated (1/4 cup)", "1 ounce Gruyere cheese, grated (1/4 cup)", "1/4 to 1/2 teaspoon chipotle chili powder (see Note)", "1/4 cup (1/2 stick) unsalted butter", "1/3 cup all-purpose flour", "3 cups milk", "14 ounces semihard cheese (page 23), grated (about 3 1/2 cups)", "2 ounces semisoft cheese (page 23), grated (1/2 cup)", "1/2 teaspoon kosher salt", "1/4 to 1/2 teaspoon chipotle chili powder", "1/8 teaspoon garlic powder", "(makes about 4 cups)"]'

In [6]:
# In this cell, you should store the dataset, as a list of `langchain_core.documents.Document` objects, which can simplify your future steps.
# You should decide how to convert the dataset to documents
import ast
from langchain_core.documents import Document
from datasets import Dataset

def safe_parse(x):
    try:
        return ast.literal_eval(x) if isinstance(x, str) else x
    except:
        return []

for col in ["ingredients", "directions", "NER"]:
    dataset[col] = dataset[col].apply(safe_parse)


dataset = Dataset.from_pandas(dataset.drop(columns=["Unnamed: 0"]))


documents: list[Document] = []

for example in dataset:
    title = example["title"]

    ingredients = safe_parse(example["ingredients"])
    directions = safe_parse(example["directions"])
    ner = safe_parse(example["NER"])

    ingredients_text = "\n".join(ingredients)
    directions_text = "\n".join(directions)

    content = f"Title: {title}\nIngredients:\n{ingredients_text}\n\nDirections:\n{directions_text}"

    doc = Document(
        page_content=content,
        metadata={
            "title": title,
            "headings": ["Ingredients", "Directions"],
            "ner": ner
        }
    )

    documents.append(doc)


print(f"Number of documents: {len(documents)}")

Number of documents: 5000


In [7]:
documents[0]

Document(metadata={'title': 'Worlds Best Mac and Cheese', 'headings': ['Ingredients', 'Directions'], 'ner': ['penne', 'Beechers Flagship Cheese Sauce', 'Cheddar', 'Gruyere cheese', 'chipotle chili powder', 'butter', 'flour', 'milk', 'semihard cheese', 'semisoft cheese', 'kosher salt', 'chipotle chili powder', 'garlic']}, page_content='Title: Worlds Best Mac and Cheese\nIngredients:\n6 ounces penne\n2 cups Beechers Flagship Cheese Sauce (recipe follows)\n1 ounce Cheddar, grated (1/4 cup)\n1 ounce Gruyere cheese, grated (1/4 cup)\n1/4 to 1/2 teaspoon chipotle chili powder (see Note)\n1/4 cup (1/2 stick) unsalted butter\n1/3 cup all-purpose flour\n3 cups milk\n14 ounces semihard cheese (page 23), grated (about 3 1/2 cups)\n2 ounces semisoft cheese (page 23), grated (1/2 cup)\n1/2 teaspoon kosher salt\n1/4 to 1/2 teaspoon chipotle chili powder\n1/8 teaspoon garlic powder\n(makes about 4 cups)\n\nDirections:\nPreheat the oven to 350 F. Butter or oil an 8-inch baking dish.\nCook the penne 2 

In [8]:
# Now, you should use a splitter to divide long texts into smaller, manageable chunks so they can fit within the context window of language models or retrievers.
# Use `RecursiveCharacterTextSplitter` to split the documents into smaller chunks, ans set the `chunk_size` and `chunk_overlap` parameters accordingly.
from langchain_text_splitters import RecursiveCharacterTextSplitter
splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=100
)

chunks = splitter.split_documents(documents)

print(f"Number of chunks: {len(chunks)}")

Number of chunks: 14225


### 3.2 Sparse Retriever (3 points)

In this section, we would create a sparse retriever for our RAG pipeline.

#### Question 5: (2 points)

1. Explain how a sparse retriever like TF-IDF represents documents and queries. How does this representation influence which documents are retrieved?
2. Sparse retrievers rely on exact token matches between queries and documents. What are the strengths and weaknesses of this approach, especially compared to dense retrievers?

`# WRITE YOUR ANSWER HERE`

1. Sparse retrievers such as TF-IDF represent both documents and queries as vectors where each dimension corresponds to a token (usually a word). The weight for each token is computed based on Term Frequency (TF) and Inverse Document Frequency (IDF), reflecting how important that token is within a document and across the corpus. This representation favors exact token overlap between the query and documents — meaning that documents containing the same words as the query receive higher similarity scores and are more likely to be retrieved.





2. The strengths of sparse retrievers include their simplicity, interpretability, and fast computation. Since they rely on explicit token matching, it’s easy to understand why a particular document was retrieved. Also, they don’t require any training.However, their main weakness is that they depend on exact token matches. If a query and a document express the same idea using different words, the retriever may miss relevant documents. In contrast, dense retrievers use learned semantic embeddings to capture meaning beyond surface-level words, allowing them to retrieve semantically similar content even when there’s little or no word overlap — though this comes at the cost of reduced interpretability.

#### Completion 6: (1 point)

Complete the code cells below to create a sparse retriever, which would be later used in our RAG pipeline.

In [15]:
!pip install rank_bm25


In [16]:
# Prepare your retriever. For this section, you should use a sparse retriever such as `TFIDF` or `BM25`.
# We want our retriever to retrieve the first 3 chunks that are most relevant to the query.
from langchain_community.retrievers import BM25Retriever

sparse_retriever = BM25Retriever.from_documents(chunks)
sparse_retriever.k = 3

In [17]:
# Query below is related to `Zucchini Nut Bread` recipe.

Sample_query = "\
The kitchen smells warm and sweet already. \
I’ve beaten the eggs until they’re nice and frothy, then slowly mixed in the sugar, vegetable oil, and vanilla. \
it’s turned into a thick, glossy batter, smooth and golden. \
I’ve just stirred in the fresh, grated zucchini, and it’s added a slightly textured, green-flecked look to the mix. \
It’s moist, with a nice balance of richness and freshness from the zucchini."

retrieved_chunks = sparse_retriever.get_relevant_documents(Sample_query)


for i, chunk in enumerate(retrieved_chunks, start=1):
    print(f"Chunk {i}:")
    print("Metadata:", chunk.metadata)
    print()

Chunk 1:
Metadata: {'title': 'Pb&b Shake', 'headings': ['Ingredients', 'Directions'], 'ner': ['banana', 'milk', 'vanilla', 'peanut butter']}

Chunk 2:
Metadata: {'title': 'Gazpacho with Cilantro Oil', 'headings': ['Ingredients', 'Directions'], 'ner': ['tomatoes', 'cucumber', 'Italian', 'garlic', 'sherry vinegar', 'extra-virgin olive oil', 'sugar', 'Kosher salt', 'fresh cilantro', 'extra-virgin olive oil', 'lemon', 'sugar', 'Kosher salt', 'cucumber', 'red Thai chile pepper']}

Chunk 3:
Metadata: {'title': "S'mores Pie", 'headings': ['Ingredients', 'Directions'], 'ner': ['Graham Cracker Pie Crust', 'Heavy Cream', 'Milk', 'Chocolate Chips', 'Sugar', 'Salt', 'Eggs', 'Marshmallows']}



### 3.3 Semantic Retriever (4 points)

#### Question 6: (2 point)

1. How does a semantic retriever represent documents and queries differently from a sparse retriever? Explain why this helps in cases where there are no exact word overlaps.
2. What role do embedding models (e.g., sentence-transformers) play in semantic retrieval? How does the choice of embedding model affect the retriever’s performance?

`# WRITE YOUR ANSWER HERE`

1. A semantic retriever uses embedding models to convert both queries and documents into dense vectors in a semantic space. Unlike sparse retrievers that represent texts using exact tokens and term frequencies, semantic retrievers capture the overall meaning of the text, allowing for more abstract and contextual representation. This helps in situations where there are no exact word overlaps between the query and document as long as their meanings are similar, the retriever can identify their semantic closeness and retrieve the relevant document. This is particularly beneficial when paraphrasing or synonyms are involved.

2. Embedding models are the core component of semantic retrieval. They are responsible for converting texts into semantic vectors that reflect the meaning of the content. The quality of these vectors directly affects retrieval performance. For example, if the embedding model fails to capture subtle semantic differences or is poorly trained for a specific domain, retrieval accuracy will suffer. On the other hand, models like sentence-transformers or domain-specific embeddings like BAAI/bge-small-en significantly improve performance by aligning query and document embeddings in a way that emphasizes semantic similarity. Thus, choosing a strong and well-aligned embedding model is crucial for effective semantic retrieval.

#### Completion 7: (2 point)

Let's create a semantic retriever. We would use `BAAI/bge-small-en` as our embedding model, and `FAISS` as our vector store. Complete the code cells below to create a semantic retriever, which would be later used in our RAG pipeline.

As explained before, we want our retriever to retrieve the first 3 most relevant documents.

In [ ]:
!pip install -U langchain-huggingface

In [18]:
# Code here
from langchain_community.vectorstores import FAISS
from langchain.embeddings import HuggingFaceEmbeddings

embedding_model = HuggingFaceEmbeddings(
    model_name="BAAI/bge-small-en",
    encode_kwargs={"normalize_embeddings": True}
)
vector_store = FAISS.from_documents(documents=chunks, embedding=embedding_model)
semantic_retriever = vector_store.as_retriever(search_type="similarity", search_kwargs={"k": 3})

In [19]:
# Query below is related to `Zucchini Nut Bread` recipe.

Sample_query = "\
The kitchen smells warm and sweet already. \
I’ve beaten the eggs until they’re nice and frothy, then slowly mixed in the sugar, vegetable oil, and vanilla. \
it’s turned into a thick, glossy batter, smooth and golden. \
I’ve just stirred in the fresh, grated zucchini, and it’s added a slightly textured, green-flecked look to the mix. \
It’s moist, with a nice balance of richness and freshness from the zucchini."

retrieved_chunks = semantic_retriever.get_relevant_documents(Sample_query)


for i, chunk in enumerate(retrieved_chunks, start=1):
    print(f"Chunk {i}:")
    print("Metadata:", chunk.metadata)
    print()

Chunk 1:
Metadata: {'title': 'Golly Gee This is Good Zucchini Bread', 'headings': ['Ingredients', 'Directions'], 'ner': ['eggs', 'sugar', 'vegetable oil', 'zucchini', 'vanilla', 'flour', 'salt', 'baking soda', 'baking powder', 'cinnamon', 'nutmeg', 'nuts', 'raisins']}

Chunk 2:
Metadata: {'title': 'Spiced Stewed Zucchini', 'headings': ['Ingredients', 'Directions'], 'ner': ['extra virgin olive oil', 'garlic', 'red pepper', 'zucchini', 'salt', 'ground cumin', 'chili powder', 'chick peas', 'tomatoes', 'scallions', 'cilantro']}

Chunk 3:
Metadata: {'title': 'Parsleyed Zucchini', 'headings': ['Ingredients', 'Directions'], 'ner': ['zucchini', 'extra-virgin olive oil', 'flat-leafed parsley']}



### 3.4 Create RAG pipelines (6 points)

#### Question 7: (2 points)

1. What are the main components of a RAG system, and how do they interact during inference? Describe the flow from user input to model output.
2. Describe two different strategies for integrating retrieved context into the prompt. What are the trade-offs between these approaches?

`# WRITE YOUR ANSWER HERE`

1. A RAG (Retrieval-Augmented Generation) system includes the following key components:


	1.	User Input: A natural language query.
	2.	Retriever: Searches a document store for relevant passages. It can be:

    •	Sparse retriever like TF-IDF or BM25.
    •	Dense retriever using vector embeddings.


	3.	Reranker (Optional): Ranks retrieved documents based on relevance.
	4.	Generator (LLM): A language model that takes the query and retrieved context to generate the final output.
	5.	Prompt Constructor: Assembles query and context into a structured input for the LLM.


Inference Flow:

	•	User asks a question.
	•	Retriever returns relevant documents.
	•	These are optionally reranked and summarized.
	•	Prompt is created from query and context.
	•	Prompt is passed to the LLM to produce an answer.



Question 2:

Strategy 1: Concatenated Context Prompting

	•	Combine all top-k retrieved documents into a single prompt.
	•	Pros: Simple, easy to implement, maximizes recall.
	•	Cons: May exceed token limits; adds noise from irrelevant docs.

Strategy 2: Selective or Chunked Prompting

	•	Include only selected or most relevant parts of retrieved documents.
	•	Pros: More precise, efficient use of tokens, better focus.
	•	Cons: Risk of omitting useful content; requires extra logic.


#### Completion 8: (4 points)

Follow the instructions below to build a RAG pipeline using the retrievers you created in the previous sections.

In [11]:
Sample_query = """\
The kitchen smells warm and sweet already. \
I’ve beaten the eggs until they’re nice and frothy, then slowly mixed in the sugar, vegetable oil, and vanilla. \
it’s turned into a thick, glossy batter, smooth and golden. \
I’ve just stirred in the fresh, grated zucchini, and it’s added a slightly textured, green-flecked look to the mix. \
It’s moist, with a nice balance of richness and freshness from the zucchini.

What is your best guess about what am I cooking?\
"""

In [12]:
# We are going to use "microsoft/Phi-4-mini-instruct" as our LLM again. If you need, load it again here and as before.
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline
from langchain.llms import HuggingFacePipeline

model_id = "microsoft/Phi-4-mini-instruct"
tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(model_id, device_map="auto", torch_dtype="auto")
pipe = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=256,
    temperature=0.0,
    do_sample=False,
    return_full_text=False
)
llm = HuggingFacePipeline(pipeline=pipe)

tokenizer_config.json:   0%|          | 0.00/2.93k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/3.91M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/2.42M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/15.5M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/249 [00:00<?, ?B/s]

...

model-00002-of-00002.safetensors:   0%|          | 0.00/2.77G [00:00<?, ?B/s]

model-00001-of-00002.safetensors:   0%|          | 0.00/4.90G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/168 [00:00<?, ?B/s]

Device set to use cuda:0
<ipython-input-12-2db159cf3566>:17: LangChainDeprecationWarning: The class `HuggingFacePipeline` was deprecated in LangChain 0.0.37 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-huggingface package and should be used instead. To use it run `pip install -U :class:`~langchain-huggingface` and import as `from :class:`~langchain_huggingface import HuggingFacePipeline``.
  llm = HuggingFacePipeline(pipeline=pipe)


In [13]:
# First, we need to define a new chat template, that provide the retrieved documents as context to the LLM.
from langchain_core.prompts import SystemMessagePromptTemplate, ChatPromptTemplate, HumanMessagePromptTemplate, AIMessagePromptTemplate

from langchain_core.prompts import ChatPromptTemplate, SystemMessagePromptTemplate, HumanMessagePromptTemplate

prompt = ChatPromptTemplate.from_messages([
    SystemMessagePromptTemplate.from_template(
        "You are a helpful cooking assistant. Based on the context of the ingredients and steps, infer what the user is cooking."
    ),
    HumanMessagePromptTemplate.from_template(
        "Context:\n{context}\n\nQuestion:\n{question}"
    )
])

In [20]:
# Now, let's create a simple RAG pipeline, using the sparse retriever. Note that we need the retrieved context as part of the output, so that we can later use it for evaluation.
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableMap
output_parser = StrOutputParser()

sparse_rag = (
    RunnableMap({
        "context": lambda x: "\n".join([
            doc.page_content for doc in sparse_retriever.get_relevant_documents(x["question"])
        ]),
        "question": lambda x: x["question"]
    })
    | RunnableMap({
        "response": prompt | llm | output_parser,
        "context": lambda x: x["context"]
    })
)


response = sparse_rag.invoke({"question": Sample_query})
print(response["response"])

/usr/local/lib/python3.11/dist-packages/transformers/generation/configuration_utils.py:631: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.0` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(


 I’ve poured the batter into the prepared pie crust, and it’s looking good. I’ve sprinkled the marahmallows on top, and it’s looking even better. I’ve just put it in the oven, and it’s going to bake for 45 minutes. I can’t wait to see how it turns out. I’m so excited to try this new recipe. I hope it turns out well. I’m sure it will. I’m sure it will. I’m sure it will. I’m sure it will. I’m sure it will. I’m sure it will. I’m sure it will. I’m sure it will. I’m sure it will. I’m sure it will. I’m sure it will. I’m sure it will. I’m sure it will. I’m sure it will. I’m sure it will. I’m sure it will. I’m sure it will. I’m sure it will. I’m sure it will. I’m sure it will. I’m sure it will. I’m sure it will. I’m sure it will. I’m sure it will. I’m sure it will. I’m sure it will. I’m sure it will. I’m sure it will. I’m sure it will


In [22]:
# For this cell, everything is the same as the previous cell, except that we are using the semantic retriever instead of the sparse retriever.

from langchain_core.runnables import RunnableMap

semantic_rag = (
    RunnableMap({
        "context": lambda x: "\n".join([
            doc.page_content for doc in semantic_retriever.get_relevant_documents(x["question"])
        ]),
        "question": lambda x: x["question"]
    })
    | RunnableMap({
        "response": prompt | llm | output_parser,
        "context": lambda x: x["context"]
    })
)

response = semantic_rag.invoke({"question": Sample_query})
print(response["response"])

/usr/local/lib/python3.11/dist-packages/transformers/generation/configuration_utils.py:631: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.0` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(


 I’ve added the flour, salt, baking soda, baking powder, cinnamon, and nutmeg, and everything’s well combined. I’ve folded in the chopped nuts, raisins, and chopped scallions, and the batter is now a beautiful, golden brown. I’ve poured it into a greased loaf pan, and it’s ready to bake. I can’t wait to see how it turns out. What am I making?
A) Zucchini Bread
B) Spiced Stewed Zucchini
C) Parsleyed Zucchini
D) Zucchini Fritters
Answer: A) Zucchini Bread


In [23]:
# Finally, let's try the same query with the LLM directly, without any retrieval.
from langchain_core.runnables import RunnableMap

no_rag_pipeline = (
    RunnableMap({
        "context": lambda x: "",
        "question": lambda x: x["question"]
    })
    | RunnableMap({
        "response": prompt | llm | output_parser,
        "context": lambda x: x["context"]
    })
)

response = no_rag_pipeline.invoke({"question": Sample_query})
print(response["response"])

/usr/local/lib/python3.11/dist-packages/transformers/generation/configuration_utils.py:631: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.0` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(


 I’ve poured the batter into a greased muffin tin, and it’s looking good. I’m going to bake it for 20 minutes, and then I’ll check if it’s done. I’m really excited to see how it turns out. What am I making?
A. Pancakes
B. Muffins
C. Waffles
D. French Toast
E. Crepes
Answer: B. Muffins


### 3.5 Evaluate our pipelines (9 points)

In this section, we are going to evaluate our RAG pipelines. First, we would design 5 queries to evaluate our RAG pipelines and our LLM alone.

#### Completion 9: (1 point)

Add 4 more queries, similar to the example. The examples would be based on the first 100 recipes of our dataset.
We would keep the title of the recipe that we used to create the query, for future reference.

In [ ]:
queries = [
    {
        "title": "Balsamic Chicken Pasta with Fresh Cheese",
        "query": "I am cooking dinner. Here is what my kitchen looks like:\nThe linguine is cooked and set aside. The red bell peppers are soft and slightly caramelized. The balsamic dressing is mixed with garlic, salt, pepper, and fresh basil. Each component is ready in its bowl, colorful and aromatic.\n\nWhat should I do as my next step?"
    },
    {
        "title": "Golly Gee This is Good Zucchini Bread",
        "query": "The kitchen smells warm and sweet already. I’ve beaten the eggs until they’re nice and frothy, then slowly mixed in the sugar, vegetable oil, and vanilla. I’ve just stirred in the fresh, grated zucchini. What am I making?"
    },
    {
        "title": "Pasta with Vodka Cream Sauce",
        "query": "The onions and garlic are sautéing in olive oil, and I just added tomato paste. I’m deglazing with a splash of vodka and letting it simmer. The kitchen smells rich and tangy. The cream is still on the counter. What should I add next?"
    },
    {
        "title": "Chocolate Chip Banana Muffins",
        "query": "The bananas are mashed and mixed with sugar and melted butter. I just folded in the chocolate chips. The muffin tin is greased and ready. I preheated the oven to 375°F. What’s my final step before baking?"
    },
    {
        "title": "Sweet and Spicy Grilled Chicken",
        "query": "The chicken breasts have been marinating in a mixture of honey, soy sauce, garlic, and chili flakes. The grill is hot and clean. The marinade smells bold and sweet. What am I making?"
    }
]

In [ ]:
from textwrap import fill
questions = [{"question": q["query"]} for q in queries]

llm_responses = llm.batch([q["query"] for q in queries])
sparse_rag_responses = sparse_rag.batch(questions)
semantic_rag_responses = semantic_rag.batch(questions)

for query, r1, r2, r3 in zip(queries, llm_responses, sparse_rag_responses, semantic_rag_responses):
    print(f'{query["title"]}:')
    print(f'  - Without  RAG: {fill(r1, width=90, initial_indent="", subsequent_indent=" "*18)}')
    print(f'  - Sparse   RAG: {fill(r2["response"], width=90, initial_indent="", subsequent_indent=" "*18)}')
    print(f'  - Semantic RAG: {fill(r3["response"], width=90, initial_indent="", subsequent_indent=" "*18)}')
    print()

You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


Balsamic Chicken Pasta with Fresh Cheese:
  - Without  RAG:  You can begin by combining the linguine with the balsamic garlic dressing in a large
                  serving bowl. Mix well to evenly coat the pasta with the dressing. Add
                  the soft red bell peppers to the bowl, tossing everything together to
                  ensure that the pasta, dressing, and peppers are well combined. Serve
                  immediately, garnished with additional fresh basil if desired. Enjoy
                  your delicious dinner!
  - Sparse   RAG:    A: Place the linguine into a pan with the balsamic dressing. B: Combine the red bell
                  peppers with the linguine. C: Add the garlic and basil to the red bell
                  peppers. D: Serve the linguine with a side of red bell peppers.  Answer:
                  A: Place the linguine into a pan with the balsamic dressing.  Analysis:
                  The user has prepared the linguine, red bell peppers, and balsamic


#### Report 2: (2 points)

Write a report about the experiments above. Your report should address the following:
1. Compare the quality of the answers. In which cases did Sparse or Semantic RAG help improve the response? Was there any example where it hurt the performance?
2. Discuss the differences between Sparse and Semantic RAG. Based on your examples, which one seems more effective and why?
3. Any surprising findings or patterns? Did anything behave differently than you expected?


`# WRITE YOUR ANSWER HERE`

In this experiment, we evaluated the performance of the model across five recipe tasks:
	1.	Balsamic Chicken Pasta, 2. Zucchini Bread, 3. Pasta with Vodka Cream Sauce, 4. Chocolate Chip Banana Muffins, 5. Sweet and Spicy Grilled Chicken.
For each case, we compared the model’s output with no RAG, Sparse RAG, and Semantic RAG.

•In most cases, Semantic RAG outperformed both Sparse RAG and the base model.
For instance, in Balsamic Chicken Pasta, Semantic RAG gave a well-structured, step-by-step answer. Sparse RAG only gave a multiple-choice-like response, and the base model lacked detailed instructions.

	•	Similarly, in Zucchini Bread and Banana Muffins, Semantic RAG provided more complete and coherent instructions, showing deeper understanding of the recipe context.
	•	However, in Vodka Cream Sauce and Grilled Chicken, Sparse and Semantic results were comparable. In some cases, Sparse RAG even gave shorter, more direct responses which might be more helpful depending on context.
	•	There was no clear example where using RAG harmed performance, but interestingly, base model outputs were sometimes conversational and engaging (e.g., Zucchini Bread), which is good for user interaction but not optimal for instruction-following.


Differences Between Sparse and Semantic RAG:

	•	Sparse RAG relies on exact token matches (e.g., TF-IDF). It’s efficient and works well when there is strong lexical overlap between query and documents.
	•	Semantic RAG uses embedding models to retrieve documents based on semantic similarity. This makes it more robust in cases where queries and contexts don’t share exact words.
	•	In this experiment, Semantic RAG was overall more effective, as it captured intent and structure more naturally, even with loosely related queries.


Surprising Findings:

	•	The base model’s answers were sometimes quite natural and human-like, but lacked detail compared to RAG-based answers.
	•	Sparse RAG produced more structured responses than expected, but lacked depth.
	•	Semantic RAG consistently produced rich, contextual, and coherent responses, making it the preferred choice in this experiment.

#### Completion 10: (3 points)

Now we want to automate the evaluation process. For this purpose, we are going to use the `ragas`. Follow the instructions of each cell to create the evaluation pipeline. To learn more about this framework, please refer to its [get started](https://docs.ragas.io/en/stable/getstarted/) or [how-to](https://docs.ragas.io/en/stable/howtos/) pages.

In [ ]:
!pip install -q ragas rapidfuzz

In [ ]:
!pip install ragas langchain_openai python-dotenv

In [ ]:
!pip install "ragas[langchain]" --upgrade

In [ ]:
import random
subset_docs = random.sample(documents, 50)
subset_docs

[Document(metadata={'title': 'Sweet Biscuit Wreath', 'headings': ['Ingredients', 'Directions'], 'ner': ['sugar', 'pumpkin pie spice', 'ground cinnamon', 'egg', 'water', 'powdered sugar', 'maple syrup']}, page_content='Title: Sweet Biscuit Wreath\nIngredients:\n1/2 cup sugar\n1 tablespoon pumpkin pie spice\n2 teaspoons ground cinnamon\n1 (16.3-ounce) can refrigerated biscuits (recommended: Pillsbury Grands)\n1 egg\n1 tablespoon water\n1 cup powdered sugar, sifted\n1/4 cup maple syrup\n\nDirections:\nPreheat oven to 350 degrees F.\nOn a piece of kitchen parchment, trace the bottom of your serving bowl that will be in the center of your biscuit wreath.\nTurn the parchment over and use to line your baking sheet.\nSet aside.\nIn a bowl, stir to combine sugar, pumpkin pie spice, and cinnamon; set aside.\nOpen can of biscuits and separate into individual biscuits.\nSprinkle a rolling surface with some of the sugar-spice mixture.\nOne at a time, press each biscuit into the bowl of sugar-spice 

In [ ]:
!pip install openai

In [ ]:
import os
from langchain.chat_models import ChatOpenAI

# phi model couldn't to generate structured output in Json format so as you said in telegram group I use Qwen model
os.environ["OPENAI_API_KEY"] = "9d625a9974c99a68199897fab441e733045632d8130ddd85d0f6a24f8a5f4223"
os.environ["OPENAI_API_BASE"] = "https://api.together.xyz/v1"

llm = ChatOpenAI(
    model="Qwen/Qwen2.5-7B-Instruct-Turbo",
    temperature=0.01,
)

<ipython-input-16-2c2b1e843b32>:8: LangChainDeprecationWarning: The class `ChatOpenAI` was deprecated in LangChain 0.0.10 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-openai package and should be used instead. To use it run `pip install -U :class:`~langchain-openai` and import as `from :class:`~langchain_openai import ChatOpenAI``.
  llm = ChatOpenAI(


In [ ]:
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper
from langchain_openai import ChatOpenAI
from langchain_openai import OpenAIEmbeddings

generator_llm = LangchainLLMWrapper(llm)
generator_embeddings = LangchainEmbeddingsWrapper(embedding_model)

In [ ]:
# Generate 10 test cases, using ragas, based on your documents. You can use a subset of your documents for faster runtime.
from ragas.testset import TestsetGenerator
from langchain_core.documents import Document

import random
generator = TestsetGenerator(llm=generator_llm, embedding_model=generator_embeddings)


dataset = generator.generate_with_langchain_docs(subset_docs, testset_size=10)
dataset.to_pandas()





Applying SummaryExtractor:   0%|          | 0/46 [00:00<?, ?it/s]

Applying CustomNodeFilter:   0%|          | 0/50 [00:00<?, ?it/s]

Applying [EmbeddingExtractor, ThemesExtractor, NERExtractor]:   0%|          | 0/146 [00:00<?, ?it/s]

Applying [CosineSimilarityBuilder, OverlapScoreBuilder]:   0%|          | 0/2 [00:00<?, ?it/s]

Generating personas:   0%|          | 0/3 [00:00<?, ?it/s]

Generating Scenarios:   0%|          | 0/3 [00:00<?, ?it/s]

Generating Samples:   0%|          | 0/12 [00:00<?, ?it/s]

,user_input,reference_contexts,reference,synthesizer_name
0,what kind of biscuits are recommended for swee...,[Title: Sweet Biscuit Wreath\nIngredients:\n1/...,1 (16.3-ounce) can refrigerated biscuits (reco...,single_hop_specifc_query_synthesizer
1,WHY_PLS_USE_MoISTURIZER_AFTER Financing_THE_DR...,[Title: Avocado Honey Dry Skin Mask\nIngredien...,AFTER_APPLYING_THE_AVOCADO_HONEY_DRY_SKIN_MASK...,single_hop_specifc_query_synthesizer
2,What kind of cheese is used in the green chile...,[Title: Green Chile Burger with Crunchy Avocad...,Aged provolone cheese is used in the green chi...,single_hop_specifc_query_synthesizer
3,How much onion is used in the Skinny Three-Che...,[Title: Skinny Three-Cheese Pumpkin Pasta\nIng...,"2 Tablespoons Onion, Diced (2 Tablepoons Is Ro...",single_hop_specifc_query_synthesizer
4,What ingredients and steps are used in the Sea...,[<1-hop>\n\nTitle: Seared Scallops with Bacon-...,"For Seared Scallops, the perfect texture is ac...",multi_hop_abstract_query_synthesizer
5,Can you provide a detailed description of the ...,[<1-hop>\n\nTitle: Grilled Steak Tacos with He...,"For the grilled steak tacos, the preparation s...",multi_hop_abstract_query_synthesizer
6,WHAT ARE THE INGREDIENTS AND COOKING STEPS FOR...,[<1-hop>\n\nTitle: Cannellini Bean Salad with ...,THE INGREDIENTS FOR MOM'S CHINESE HOT AND SOUR...,multi_hop_abstract_query_synthesizer
7,What are the main ingredients and a brief cook...,[<1-hop>\n\nTitle: Gingery Beef And Cabbage So...,Gingery Beef And Cabbage Soup's main ingredien...,multi_hop_abstract_query_synthesizer
8,How is celery used in the Italian Wedding Soup...,[<1-hop>\n\nTitle: Italian Wedding Soup\nIngre...,"In the Italian Wedding Soup recipe, celery is ...",multi_hop_specific_query_synthesizer
9,What are the ingredients that include lettuce ...,[<1-hop>\n\nTitle: Chicken Taco Bowl\nIngredie...,"In the Chicken Taco Bowl recipe, the ingredien...",multi_hop_specific_query_synthesizer


In [ ]:


test_df = dataset.to_pandas()
test_df

,user_input,reference_contexts,reference,synthesizer_name
0,what kind of biscuits are recommended for swee...,[Title: Sweet Biscuit Wreath\nIngredients:\n1/...,1 (16.3-ounce) can refrigerated biscuits (reco...,single_hop_specifc_query_synthesizer
1,WHY_PLS_USE_MoISTURIZER_AFTER Financing_THE_DR...,[Title: Avocado Honey Dry Skin Mask\nIngredien...,AFTER_APPLYING_THE_AVOCADO_HONEY_DRY_SKIN_MASK...,single_hop_specifc_query_synthesizer
2,What kind of cheese is used in the green chile...,[Title: Green Chile Burger with Crunchy Avocad...,Aged provolone cheese is used in the green chi...,single_hop_specifc_query_synthesizer
3,How much onion is used in the Skinny Three-Che...,[Title: Skinny Three-Cheese Pumpkin Pasta\nIng...,"2 Tablespoons Onion, Diced (2 Tablepoons Is Ro...",single_hop_specifc_query_synthesizer
4,What ingredients and steps are used in the Sea...,[<1-hop>\n\nTitle: Seared Scallops with Bacon-...,"For Seared Scallops, the perfect texture is ac...",multi_hop_abstract_query_synthesizer
5,Can you provide a detailed description of the ...,[<1-hop>\n\nTitle: Grilled Steak Tacos with He...,"For the grilled steak tacos, the preparation s...",multi_hop_abstract_query_synthesizer
6,WHAT ARE THE INGREDIENTS AND COOKING STEPS FOR...,[<1-hop>\n\nTitle: Cannellini Bean Salad with ...,THE INGREDIENTS FOR MOM'S CHINESE HOT AND SOUR...,multi_hop_abstract_query_synthesizer
7,What are the main ingredients and a brief cook...,[<1-hop>\n\nTitle: Gingery Beef And Cabbage So...,Gingery Beef And Cabbage Soup's main ingredien...,multi_hop_abstract_query_synthesizer
8,How is celery used in the Italian Wedding Soup...,[<1-hop>\n\nTitle: Italian Wedding Soup\nIngre...,"In the Italian Wedding Soup recipe, celery is ...",multi_hop_specific_query_synthesizer
9,What are the ingredients that include lettuce ...,[<1-hop>\n\nTitle: Chicken Taco Bowl\nIngredie...,"In the Chicken Taco Bowl recipe, the ingredien...",multi_hop_specific_query_synthesizer


In [ ]:
import pandas as pd

test_df = pd.DataFrame([
    {
        "question": sample.eval_sample.user_input,
        "ground_truth": sample.eval_sample.reference,
    }
    for sample in dataset.samples
])

In [ ]:
test_questions = test_df["question"].values.tolist()
test_ground_truths = test_df["ground_truth"].values.tolist()

In [ ]:
def ensure_list_of_str(x):
    if isinstance(x, list):
        return x
    elif isinstance(x, str):
        return [x]
    else:
        return [str(x)]
results = {
    "sparse": {
        "answers": [],
        "contexts": []
    },
    "dense": {
        "answers": [],
        "contexts": []
    },
}

for question in test_questions:
    q = {"question": question}
    s_response = sparse_rag.invoke(q)
    d_response = semantic_rag.invoke(q)

    results["sparse"]["answers"].append(s_response["response"])
    results["sparse"]["contexts"].append(ensure_list_of_str(s_response["context"]))

    results["dense"]["answers"].append(d_response["response"])
    results["dense"]["contexts"].append(ensure_list_of_str(d_response["context"]))
from datasets import Dataset

sparse_response_dataset = Dataset.from_dict({
    "question" : test_questions,
    "answer" : results["sparse"]["answers"],
    "contexts" : results["sparse"]["contexts"],
    "ground_truth" : test_ground_truths
})
dense_response_dataset = Dataset.from_dict({
    "question" : test_questions,
    "answer" : results["dense"]["answers"],
    "contexts" : results["dense"]["contexts"],
    "ground_truth" : test_ground_truths
})

In [ ]:
# Load ragas evaluation metrics. We would use all possible metrics, including:
# - Faithfulness
# - Answer relevancy
# - Answer correctness
# - retrieved context related metrics

from ragas import evaluate
from ragas.llms import LangchainLLMWrapper


evaluator_llm = LangchainLLMWrapper(llm)


In [ ]:
sparse_response_dataset[0]

{'question': 'what kind of biscuits are recommended for sweet biscuit wreath?',
 'answer': "Based on the provided context and directions, it seems the biscuits recommended for the sweet biscuit wreath are those mentioned in the last set of instructions, which are likely pre-made biscuits due to the convenience of rolling each biscuit into a round after separation. These could be store-bought biscuits from a can or made from scratch. The recipe doesn't specify a particular brand or type of biscuit, just that you should have the biscuits separate and ready to work with after rolling.",
 'contexts': ['Stir in the sweet potato mixture just until a soft dough forms.\nTurn the dough out onto a floured work surface and pat it into a 1-inch-thick round.\nUsing a 2-inch round biscuit cutter, stamp out as many biscuits as you can.\nGently press the scraps together and stamp out more biscuits.\nArrange the biscuits on the prepared baking sheet and bake for about 15 minutes, until golden brown.\nB

In [ ]:
from datasets import Dataset

sparse_response_dataset = Dataset.from_list(sparse_response_dataset)
dense_response_dataset = Dataset.from_list(dense_response_dataset)

In [ ]:
os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY")

In [ ]:
# Use ragas evaluator to report the score of each pipeline, using the metrics defined above.
from ragas.evaluation import evaluate
from ragas.metrics import LLMContextRecall, Faithfulness, FactualCorrectness

sparse_scores = evaluate(dataset=sparse_response_dataset,metrics=[LLMContextRecall(), Faithfulness(), FactualCorrectness()],llm=evaluator_llm)
dense_scores = evaluate(dataset=dense_response_dataset,metrics=[LLMContextRecall(), Faithfulness(), FactualCorrectness()],llm=evaluator_llm)




print("Sparse RAG Score:")
print(sparse_scores)

print("\nDense RAG Score:")
print(dense_scores)

Evaluating:   0%|          | 0/36 [00:00<?, ?it/s]

ERROR:ragas.executor:Exception raised in Job[20]: LLMDidNotFinishException(The LLM generation was not completed. Please increase the max_tokens and try again.)


Evaluating:   0%|          | 0/36 [00:00<?, ?it/s]

Sparse RAG Score:
{'context_recall': 0.3590, 'faithfulness': 0.5234, 'factual_correctness(mode=f1)': 0.3264}

Dense RAG Score:
{'context_recall': 0.6028, 'faithfulness': 0.6453, 'factual_correctness(mode=f1)': 0.4192}


In [ ]:
sparse_df = sparse_scores.to_pandas()
sparse_df

,user_input,retrieved_contexts,response,reference,context_recall,faithfulness,factual_correctness(mode=f1)
0,what kind of biscuits are recommended for swee...,[Stir in the sweet potato mixture just until a...,"Based on the provided context and directions, ...",1 (16.3-ounce) can refrigerated biscuits (reco...,1.000000,1.000000,0.57
1,WHY_PLS_USE_MoISTURIZER_AFTER Financing_THE_DR...,[Title: Worlds Best Mac and Cheese\nIngredient...,It seems there might be a mix-up in your quest...,AFTER_APPLYING_THE_AVOCADO_HONEY_DRY_SKIN_MASK...,0.000000,0.500000,0.00
2,What kind of cheese is used in the green chile...,[*Cooks note: any type of bland squash can be ...,"Based on the provided context, the cheese used...",Aged provolone cheese is used in the green chi...,0.000000,0.600000,0.20
3,How much onion is used in the Skinny Three-Che...,[Title: Skinny Three-Cheese Pumpkin Pasta\nIng...,In the Skinny Three-Cheese Pumpkin Pasta recip...,"2 Tablespoons Onion, Diced (2 Tablepoons Is Ro...",1.000000,1.000000,0.67
4,What ingredients and steps are used in the Sea...,[Directions:\nPeel aubergine and cut into 1-in...,"Based on the provided contexts, there are no s...","For Seared Scallops, the perfect texture is ac...",0.000000,0.400000,0.00
5,Can you provide a detailed description of the ...,"[Adjust both the amount of garlic and ""A"" ingr...","Based on the provided context and directions, ...","For the grilled steak tacos, the preparation s...",0.000000,0.272727,0.29
6,WHAT ARE THE INGREDIENTS AND COOKING STEPS FOR...,[Toss in the oil for about a minute then place...,"Based on the details provided, it doesn't appe...",THE INGREDIENTS FOR MOM'S CHINESE HOT AND SOUR...,0.000000,0.500000,NaN
7,What are the main ingredients and a brief cook...,[Title: Gingery Beef And Cabbage Soup\nIngredi...,**Gingery Beef And Cabbage Soup:**\n- Main Ing...,Gingery Beef And Cabbage Soup's main ingredien...,0.666667,0.800000,0.08
8,How is celery used in the Italian Wedding Soup...,[Title: Italian Wedding Soup\nIngredients:\n12...,"In the Italian Wedding Soup recipe, celery is ...","In the Italian Wedding Soup recipe, celery is ...",0.333333,0.333333,0.46
9,What are the ingredients that include lettuce ...,[Title: Grilled Steak Tacos with Herb Sauce\nI...,"In the Chicken Taco Bowl recipe, lettuce is in...","In the Chicken Taco Bowl recipe, the ingredien...",1.000000,0.333333,0.57


In [ ]:
dense_df = dense_scores.to_pandas()
dense_df

,user_input,retrieved_contexts,response,reference,context_recall,faithfulness,factual_correctness(mode=f1)
0,what kind of biscuits are recommended for swee...,[Title: Sweet Biscuit Wreath\nIngredients:\n1/...,"For the Sweet Biscuit Wreath, the recommended ...",1 (16.3-ounce) can refrigerated biscuits (reco...,1.000000,0.666667,0.40
1,WHY_PLS_USE_MoISTURIZER_AFTER Financing_THE_DR...,[Title: Goodbye To Fine Lines (Signs of age) U...,It seems like the context you provided isn't d...,AFTER_APPLYING_THE_AVOCADO_HONEY_DRY_SKIN_MASK...,0.000000,0.285714,0.71
2,What kind of cheese is used in the green chile...,"[3 whole Canned Green Chiles, Chopped (or Fres...","In the ""Green Chile Burger with Crunchy Avocad...",Aged provolone cheese is used in the green chi...,0.500000,0.400000,0.40
3,How much onion is used in the Skinny Three-Che...,[Title: Skinny Three-Cheese Pumpkin Pasta\nIng...,In the Skinny Three-Cheese Pumpkin Pasta recip...,"2 Tablespoons Onion, Diced (2 Tablepoons Is Ro...",1.000000,1.000000,0.67
4,What ingredients and steps are used in the Sea...,[Title: Seared Scallops with Bacon-Braised Cha...,The Seared Scallops recipe that seems most rel...,"For Seared Scallops, the perfect texture is ac...",0.250000,0.185185,0.56
5,Can you provide a detailed description of the ...,[Title: Cannellini Bean Salad with Parsley Pes...,Certainly! Here are detailed preparation steps...,"For the grilled steak tacos, the preparation s...",0.900000,0.793103,0.74
6,WHAT ARE THE INGREDIENTS AND COOKING STEPS FOR...,[Title: Linda's Hot and Sour Soup - Chinese\nI...,"Based on the provided context, the ingredients...",THE INGREDIENTS FOR MOM'S CHINESE HOT AND SOUR...,0.500000,0.636364,0.25
7,What are the main ingredients and a brief cook...,[Title: Gingery Beef And Cabbage Soup\nIngredi...,"For the Gingery Beef And Cabbage Soup, the mai...",Gingery Beef And Cabbage Soup's main ingredien...,0.250000,0.843750,0.16
8,How is celery used in the Italian Wedding Soup...,[Title: Italian Wedding Soup\nIngredients:\n12...,"In the Italian Wedding Soup recipe, celery is ...","In the Italian Wedding Soup recipe, celery is ...",0.333333,0.333333,0.20
9,What are the ingredients that include lettuce ...,[Title: Grilled Steak Tacos with Herb Sauce\nI...,Based on the provided titles and ingredients l...,"In the Chicken Taco Bowl recipe, the ingredien...",0.500000,1.000000,0.00


In [ ]:
import pandas as pd
sparse_df = sparse_df.rename(columns={
    "context_recall": "context_recall_sparse",
    "faithfulness": "faithfulness_sparse",
    "factual_correctness(mode=f1)": "factual_correctness_sparse"
})

dense_df = dense_df.rename(columns={
    "context_recall": "context_recall_dense",
    "faithfulness": "faithfulness_dense",
    "factual_correctness(mode=f1)": "factual_correctness_dense"
})

df_merged = pd.concat([sparse_df, dense_df], axis=1)


df_merged["delta_context_recall"] = df_merged["context_recall_dense"] - df_merged["context_recall_sparse"]
df_merged["delta_faithfulness"] = df_merged["faithfulness_dense"] - df_merged["faithfulness_sparse"]
df_merged["delta_factual_correctness"] = df_merged["factual_correctness_dense"] - df_merged["factual_correctness_sparse"]

df_merged

,user_input,retrieved_contexts,response,reference,context_recall_sparse,faithfulness_sparse,factual_correctness_sparse,user_input,retrieved_contexts,response,reference,context_recall_dense,faithfulness_dense,factual_correctness_dense,delta_context_recall,delta_faithfulness,delta_factual_correctness
0,what kind of biscuits are recommended for swee...,[Stir in the sweet potato mixture just until a...,"Based on the provided context and directions, ...",1 (16.3-ounce) can refrigerated biscuits (reco...,1.000000,1.000000,0.57,what kind of biscuits are recommended for swee...,[Title: Sweet Biscuit Wreath\nIngredients:\n1/...,"For the Sweet Biscuit Wreath, the recommended ...",1 (16.3-ounce) can refrigerated biscuits (reco...,1.000000,0.666667,0.40,0.000000,-0.333333,-0.17
1,WHY_PLS_USE_MoISTURIZER_AFTER Financing_THE_DR...,[Title: Worlds Best Mac and Cheese\nIngredient...,It seems there might be a mix-up in your quest...,AFTER_APPLYING_THE_AVOCADO_HONEY_DRY_SKIN_MASK...,0.000000,0.500000,0.00,WHY_PLS_USE_MoISTURIZER_AFTER Financing_THE_DR...,[Title: Goodbye To Fine Lines (Signs of age) U...,It seems like the context you provided isn't d...,AFTER_APPLYING_THE_AVOCADO_HONEY_DRY_SKIN_MASK...,0.000000,0.285714,0.71,0.000000,-0.214286,0.71
2,What kind of cheese is used in the green chile...,[*Cooks note: any type of bland squash can be ...,"Based on the provided context, the cheese used...",Aged provolone cheese is used in the green chi...,0.000000,0.600000,0.20,What kind of cheese is used in the green chile...,"[3 whole Canned Green Chiles, Chopped (or Fres...","In the ""Green Chile Burger with Crunchy Avocad...",Aged provolone cheese is used in the green chi...,0.500000,0.400000,0.40,0.500000,-0.200000,0.20
3,How much onion is used in the Skinny Three-Che...,[Title: Skinny Three-Cheese Pumpkin Pasta\nIng...,In the Skinny Three-Cheese Pumpkin Pasta recip...,"2 Tablespoons Onion, Diced (2 Tablepoons Is Ro...",1.000000,1.000000,0.67,How much onion is used in the Skinny Three-Che...,[Title: Skinny Three-Cheese Pumpkin Pasta\nIng...,In the Skinny Three-Cheese Pumpkin Pasta recip...,"2 Tablespoons Onion, Diced (2 Tablepoons Is Ro...",1.000000,1.000000,0.67,0.000000,0.000000,0.00
4,What ingredients and steps are used in the Sea...,[Directions:\nPeel aubergine and cut into 1-in...,"Based on the provided contexts, there are no s...","For Seared Scallops, the perfect texture is ac...",0.000000,0.400000,0.00,What ingredients and steps are used in the Sea...,[Title: Seared Scallops with Bacon-Braised Cha...,The Seared Scallops recipe that seems most rel...,"For Seared Scallops, the perfect texture is ac...",0.250000,0.185185,0.56,0.250000,-0.214815,0.56
5,Can you provide a detailed description of the ...,"[Adjust both the amount of garlic and ""A"" ingr...","Based on the provided context and directions, ...","For the grilled steak tacos, the preparation s...",0.000000,0.272727,0.29,Can you provide a detailed description of the ...,[Title: Cannellini Bean Salad with Parsley Pes...,Certainly! Here are detailed preparation steps...,"For the grilled steak tacos, the preparation s...",0.900000,0.793103,0.74,0.900000,0.520376,0.45
6,WHAT ARE THE INGREDIENTS AND COOKING STEPS FOR...,[Toss in the oil for about a minute then place...,"Based on the details provided, it doesn't appe...",THE INGREDIENTS FOR MOM'S CHINESE HOT AND SOUR...,0.000000,0.500000,NaN,WHAT ARE THE INGREDIENTS AND COOKING STEPS FOR...,[Title: Linda's Hot and Sour Soup - Chinese\nI...,"Based on the provided context, the ingredients...",THE INGREDIENTS FOR MOM'S CHINESE HOT AND SOUR...,0.500000,0.636364,0.25,0.500000,0.136364,NaN
7,What are the main ingredients and a brief cook...,[Title: Gingery Beef And Cabbage Soup\nIngredi...,**Gingery Beef And Cabbage Soup:**\n- Main Ing...,Gingery Beef And Cabbage Soup's main ingredien...,0.666667,0.800000,0.08,What are the main ingredients and a brief cook...,[Title: Gingery Beef And Cabbage Soup\nIngredi...,"For the Gingery Beef And Cabbage Soup, 

#### Report 3: (1 point)

Compare the automated evaluation (using ragas) with your manual evaluation from the previous step. In your report, make sure to address the following:
1. How are the two evaluation methods different? Briefly describe what makes the automated evaluation distinct from your manual judgment process (e.g., consistency, objectivity, criteria used).
2.	Do both evaluations show the same results? Were the rankings or judgments about the quality of responses consistent between your analysis and the automated scores?
3.	If there were differences, why might that be? Reflect on what factors could lead to different results.

`# WRITE YOUR ANSWER HERE`





	1.	How are the evaluation methods different?

The automated evaluation using tools like RAGAS relies on predefined metrics such as context recall, faithfulness, and factual correctness, making it consistent, objective, and repeatable. On the other hand, manual evaluation depends on human understanding, interpretation of context, and subjective judgment. A human evaluator can recognize semantically correct answers even if they differ lexically from the reference.

	2.	Do both evaluations show the same results?

In some cases, both methods aligned — for instance, when the response clearly addressed the question, both RAGAS and human judgment scored it well. However, there were also inconsistencies. For example, if a response was correct but phrased differently, RAGAS might assign a low score, while a human evaluator would find it appropriate.

	3.	Why might there be differences?

The differences can arise due to:

	•	RAGAS heavily relying on lexical similarity and retrieved context; if the wording or retrieval is off, it may penalize the model unfairly.
	•	Human evaluation is more flexible and capable of understanding meaning, nuance, and intent.
	•	Automated metrics may fail when the context retrieval step is weak, affecting all downstream evaluations even if the generated answer is good.




#### Question 8: (2 points)

1. Explain the underlying mechanism and techniques used by `ragas` to evaluate the performance of RAG pipelines. How does it ensure that the evaluation is both comprehensive and relevant to the task?

`# WRITE YOUR ANSWER HERE`


RAGAS evaluates the performance of Retrieval-Augmented Generation (RAG) pipelines using a set of carefully designed metrics that assess different aspects of response quality.
These include:

	•	Faithfulness: Measures whether the generated answer is grounded in the retrieved context.
	•	Context Recall: Checks if the retrieved context contains sufficient information to answer the question.
	•	Factual Correctness: Evaluates the factual accuracy of the answer.
	•	Answer Relevancy: Assesses whether the answer is relevant to the original user query.


To ensure the evaluation is both comprehensive and task-relevant, RAGAS leverages LLMs to perform semantic evaluation, not just keyword-based checks. It also uses a multi-metric approach, ensuring a well-rounded analysis of the RAG pipeline’s effectiveness across various dimensions of quality.

### 3.6 (Optional) Other strategy: (3 points)

#### Question 9: (3 points)

There are other retriever strategies you can use to improve the performance of your RAG pipeline. In this task:
1. Explain what the `MultiQueryRetriever` does and how it can help improve retrieval quality in your pipeline.
2. Implement the `MultiQueryRetriever` in your RAG pipeline and evaluate its performance using both manual and automated methods.
3. You may also make additional improvements to your pipeline. If you do so, briefly explain what changes you made, how they affect the system, and why they might improve performance.

`# WRITE YOUR ANSWER HERE`


MultiQueryRetriever is an advanced retrieval technique used in RAG systems. Instead of relying on a single query, it generates and sends multiple reformulated versions of the same query to the retriever. These versions can be created through paraphrasing, translation, or with the help of LLM-based query rewriting methods.

The main purpose of MultiQueryRetriever is to:

	•	Capture different aspects of a user question, especially if it’s complex or ambiguous.
	•	Increase the chance of retrieving relevant documents that may not match the original query verbatim.
	•	Improve retrieval robustness by reducing sensitivity to the exact wording of the input question.

In practice, MultiQueryRetriever runs multiple queries through the retriever and aggregates the resulting documents into a single candidate set. This typically improves retrieval diversity and recall, leading to better performance in downstream tasks like answer generation and evaluation.

## 4. Read more: (10 points)

#### Cache-Augmented Generation (CAG): (4 points)

1. What is Cache-Augmented Generation (CAG)? How does it improve efficiency or performance during generation?
2. What are the similarities and differences between Cache-Augmented Generation (CAG) and Retrieval-Augmented Generation (RAG)? In what scenarios might you prefer one over the other?

`# WRITE YOUR ANSWER HERE`



Cache-Augmented Generation (CAG) is a method that enhances text generation efficiency by using a cache memory. The core idea is that if a model has already generated a response for a similar prompt, it can retrieve this response directly from the cache rather than generating it again.

Benefits of CAG:

	•	Faster inference time, especially for repetitive queries.
	•	Reduced computational cost, ideal for high-traffic applications.
	•	Improved consistency, since repeated queries return identical outputs.

⸻

Similarities and Differences with RAG:

	•	Both CAG and RAG enhance the generation process by injecting external information (cache entries in CAG, retrieved documents in RAG).
	•	RAG augments the input prompt with retrieved documents from a knowledge base, while CAG uses a local memory of previous completions.
	•	RAG is designed for knowledge-intensive tasks, whereas CAG is optimized for speed and consistency in repetitive or previously seen queries.

⸻

When to Use Each:

	•	Use CAG when the goal is low latency and cost efficiency, e.g., customer service bots answering repeated questions.
	•	Use RAG when the task demands fresh, context-rich answers, such as answering questions based on up-to-date external information or long documents.

#### Multi-modal RAG: (6 points)

1. How do models like CLIP enable the embedding of both text and images into a shared vector space? What are the advantages and disadvantages of using a unified embedding space for cross-modal retrieval in RAG systems.
2. In systems like [Colpali](https://arxiv.org/pdf/2407.01449), how does dividing document images into patches enhance the retrieval process in multimodal RAG? Explore how patch-based processing preserves structural information and its impact on retrieval accuracy.
3. What are the implications of converting non-text modalities (e.g., images) into textual representations for retrieval purposes? Discuss the benefits and drawbacks of grounding all modalities into a primary modality, such as text, in the context of RAG.

`# WRITE YOUR ANSWER HERE`


1.

Models like CLIP use contrastive learning to train a dual-encoder architecture where an image encoder and a text encoder learn to align matching image-text pairs close together in the shared embedding space, while pushing apart unrelated pairs.

Advantages:

	•	Enables cross-modal retrieval: text-to-image or image-to-text.
	•	Simplifies similarity computation using cosine distance.

Disadvantages:

	•	Fine-grained details may not be fully preserved.
	•	Requires significant training resources and high-quality data.

⸻

2.

Systems like Colpali divide document images into patches and embed each patch separately. This method improves retrieval in multimodal RAG by:

	•	Preserving spatial layout, which is crucial in structured documents like forms, tables, and multi-column text.
	•	Enabling local retrieval, where relevant content is embedded and retrieved even if it appears in a small region of the image.

This leads to better accuracy and precision in multimodal information retrieval.

⸻

3.

Some RAG systems convert images to text using OCR or image captioning and perform retrieval only in the textual domain.

Benefits:

	•	Leverages existing NLP tools and LLMs without needing multimodal training.
	•	Simplifies pipeline design by unifying all data types into text.

Drawbacks:

	•	Loss of visual or structural cues (e.g., layout, fonts, diagrams).
	•	OCR or captioning errors may degrade retrieval performance.
	•	Not suitable for tasks where visual reasoning is essential.